# RETECO Track 1a — Retrieval Pipeline on Kaggle

**EN** — This notebook runs the custom RETECO Track 1a retrieval system (BM25, dense bi-encoder, and a BM25+dense hybrid via Reciprocal Rank Fusion) on Kaggle, using a free GPU to speed up embedding encoding.

**VI** — Notebook này chạy hệ thống retrieval Track 1a (BM25, dense bi-encoder, và hybrid BM25+dense qua Reciprocal Rank Fusion) trên Kaggle, tận dụng GPU miễn phí để tăng tốc bước encode embedding.

---

**Before running / Trước khi chạy:**
1. EN: Upload the `src/` folder (8 files under `src/track1a/`) as a Kaggle Dataset, and attach it to this notebook via **Add Data**.
   VI: Upload thư mục `src/` (8 file trong `src/track1a/`) thành 1 Kaggle Dataset, gắn vào notebook này qua **Add Data**.
2. EN: Enable **GPU** (right panel → Accelerator → GPU T4 x2) and **Internet** (right panel → Internet → On).
   VI: Bật **GPU** (panel phải → Accelerator → GPU T4 x2) và **Internet** (panel phải → Internet → On).
3. EN: Update `SRC_DATASET_PATH` in Cell 1 below to match where your dataset actually mounts — check with `!find /kaggle/input -maxdepth 5` if unsure.
   VI: Sửa `SRC_DATASET_PATH` ở Cell 1 bên dưới cho đúng đường dẫn dataset của bạn — nếu không chắc, chạy `!find /kaggle/input -maxdepth 5` để xem.


## 1. Environment setup / Thiết lập môi trường

**EN** — Creates the project folder layout that `src/track1a` expects (`RETECO/`, `reteco_data/`, `runs/`, `cache/`), copies your code in, and checks that the GPU is visible.

**VI** — Dựng cấu trúc thư mục mà `src/track1a` yêu cầu (`RETECO/`, `reteco_data/`, `runs/`, `cache/`), copy code của bạn vào, và kiểm tra GPU đã sẵn sàng chưa.


In [ ]:
import os, shutil, torch

# EN: change this if your dataset mounts somewhere else (run `!find /kaggle/input -maxdepth 5` to check)
# VI: sửa lại nếu dataset của bạn mount ở path khác (chạy `!find /kaggle/input -maxdepth 5` để kiểm tra)
SRC_DATASET_PATH = "/kaggle/input/datasets/giabotrnl/track1a-src/src"

ROOT = "/kaggle/working/RETECO-project"
os.makedirs(ROOT, exist_ok=True)
os.makedirs(f"{ROOT}/runs", exist_ok=True)
os.makedirs(f"{ROOT}/cache/embeddings", exist_ok=True)

shutil.copytree(SRC_DATASET_PATH, f"{ROOT}/src", dirs_exist_ok=True)

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("src/track1a contents:", os.listdir(f"{ROOT}/src/track1a"))


## 2. (Optional) Hugging Face token / Token Hugging Face (tùy chọn)

**EN** — Not required, but avoids the "unauthenticated requests" rate-limit warning when downloading the model and dataset. Create a read-only token at huggingface.co → Settings → Access Tokens, then add it as a Kaggle secret named `HF_TOKEN` (Add-ons → Secrets). Skip this cell if you haven't set one up.

**VI** — Không bắt buộc, nhưng tránh cảnh báo rate-limit "unauthenticated requests" khi tải model và dataset. Tạo token read-only tại huggingface.co → Settings → Access Tokens, rồi thêm vào Kaggle Secrets với tên `HF_TOKEN` (Add-ons → Secrets). Bỏ qua cell này nếu bạn chưa thiết lập.


In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN set")
except Exception as e:
    print("Skipping HF token / Bỏ qua HF token:", e)


## 3. Install dependencies & clone the organizer's starter kit
## Cài dependency & clone starter kit của ban tổ chức

**EN** — `sentence-transformers` + `faiss-cpu` are needed for the dense and hybrid methods (the bm25 method needs nothing extra). We also clone the official RETECO repo to get `scorer.py` and `format_checker.py` — the organizer's own scoring and validation tools, so our results match what the leaderboard would compute.

**VI** — `sentence-transformers` + `faiss-cpu` cần cho phương pháp dense và hybrid (bm25 không cần gì thêm). Clone repo RETECO chính thức để lấy `scorer.py` và `format_checker.py` — công cụ chấm điểm và kiểm tra định dạng của ban tổ chức, để kết quả khớp với cách leaderboard sẽ tính.


In [ ]:
%pip install -q huggingface_hub sentence-transformers faiss-cpu
!git clone -q https://github.com/DataScienceUIBK/RETECO.git {ROOT}/RETECO
print("done")


## 4. Download data / Tải dữ liệu

**EN** — Starts with just two small Track 1 domains (`iota`, `law`) instead of the full 4.5GB release, so the pipeline can be validated quickly. Remove `allow_patterns` later to fetch everything once you're ready to run all 13 domains.

**VI** — Bắt đầu chỉ với 2 domain Track 1 nhỏ (`iota`, `law`) thay vì tải nguyên 4.5GB, để kiểm tra pipeline nhanh. Bỏ `allow_patterns` sau khi đã sẵn sàng chạy đủ 13 domain.


In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="DataScience-UIBK/RETECO-SemEval2027",
    repo_type="dataset",
    local_dir=f"{ROOT}/reteco_data",
    allow_patterns=["track1_tempo/iota/*", "track1_tempo/law/*", "split_manifest.json"],
)
print("data downloaded")


## 5. Run all three methods on `iota` / Chạy cả 3 phương pháp trên `iota`

**EN** — Runs BM25 (lexical only), dense (embedding-only), and hybrid (BM25+dense fused via RRF) on the same domain and split, so you can see each method's individual contribution. Each run writes a TREC run file, gets validated by the organizer's `format_checker.py`, and scored by the organizer's `scorer.py`.

**VI** — Chạy BM25 (chỉ từ khóa), dense (chỉ embedding), và hybrid (BM25+dense qua RRF) trên cùng domain/split, để thấy đóng góp riêng của từng phương pháp. Mỗi lần chạy ghi ra file TREC, được `format_checker.py` của ban tổ chức kiểm tra, và `scorer.py` chấm điểm.


In [ ]:
for method in ["bm25", "dense", "hybrid"]:
    print(f"\n########## {method.upper()} ##########")
    !cd {ROOT} && python src/track1a/run_release.py --split train --track1 iota --method {method}


## 6. Run hybrid on a second domain / Chạy hybrid trên domain thứ hai

**EN** — `iota` only has 7 training questions — too few to draw conclusions from alone. Running `law` too gives a second, independent data point before deciding whether to scale up to all 13 domains.

**VI** — `iota` chỉ có 7 câu hỏi train — quá ít để kết luận một mình. Chạy thêm `law` cho một điểm dữ liệu độc lập thứ hai trước khi quyết định có mở rộng ra đủ 13 domain hay không.


In [ ]:
!cd {ROOT} && python src/track1a/run_release.py --split train --track1 law --method hybrid


## 7. Compare all results / So sánh tất cả kết quả

**EN** — Each `run_release.py` call writes its own `results_<split>_<method>.json`, so nothing gets overwritten. This cell reads all of them back and prints a quick comparison table.

**VI** — Mỗi lần gọi `run_release.py` ghi ra file `results_<split>_<method>.json` riêng, không ghi đè nhau. Cell này đọc lại tất cả và in bảng so sánh nhanh.


In [ ]:
import json, glob

for f in sorted(glob.glob(f"{ROOT}/runs/track1a/results_*.json")):
    r = json.load(open(f))
    print(f"{os.path.basename(f):40s} macro_nDCG@10 = {r['macro_nDCG@10']}")


## 8. Save results before closing the session
## Lưu kết quả trước khi đóng session

**EN** — `/kaggle/working/` persists only if you **Save Version**. This zips the `runs/` folder so you can download it from the notebook's Output tab, or re-use it locally.

**VI** — `/kaggle/working/` chỉ được giữ lại nếu bạn **Save Version**. Cell này nén thư mục `runs/` để tải về từ tab Output của notebook, hoặc dùng lại ở máy local.


In [ ]:
!cd {ROOT} && zip -r /kaggle/working/runs_output.zip runs
print("zipped -> /kaggle/working/runs_output.zip")


---
## Next steps / Bước tiếp theo

**EN**
- If hybrid clearly beats bm25/dense alone on both domains: scale up — remove `--track1 iota`/`law` to run all 13 Track 1 domains, and drop `allow_patterns` in Cell 4 to download the full dataset.
- Only run `--split dev` once, at the very end, as a held-out check — per the competition rules, dev is not meant to be tuned against.
- Consider a cross-encoder reranker over the hybrid method's top candidates, and mining `guidance_train.jsonl` for temporal query patterns, as the next accuracy gains.

**VI**
- Nếu hybrid rõ ràng vượt trội bm25/dense riêng lẻ trên cả 2 domain: mở rộng — bỏ `--track1 iota`/`law` để chạy đủ 13 domain Track 1, và bỏ `allow_patterns` ở Cell 4 để tải toàn bộ dataset.
- Chỉ chạy `--split dev` một lần duy nhất, ở cuối cùng, như bước kiểm tra giữ lại — theo luật thi, dev không dùng để tinh chỉnh.
- Cân nhắc thêm cross-encoder reranker trên kết quả top của hybrid, và khai thác `guidance_train.jsonl` để tìm mẫu câu hỏi temporal, làm bước tăng điểm tiếp theo.
